In [2]:
from B1Py.robot.model import B1Model

In [3]:
model = B1Model()

In [4]:
import sys
import time
import math
#import numpy as np

sys.path.append('/home/rstaion/projects/rooholla/locomotion/unitree_legged_sdk/lib/python/amd64')
import robot_interface as sdk

In [5]:
d = {'FR_0':0, 'FR_1':1, 'FR_2':2,
         'FL_0':3, 'FL_1':4, 'FL_2':5, 
         'RR_0':6, 'RR_1':7, 'RR_2':8, 
         'RL_0':9, 'RL_1':10, 'RL_2':11 }
PosStopF  = math.pow(10,9)
VelStopF  = 16000.0
HIGHLEVEL = 0xee
LOWLEVEL  = 0xff
sin_mid_q = [0.0, 1.2, -2.0]
dt = 0.002
qInit = [0, 0, 0]
qDes = [0, 0, 0]
sin_count = 0
rate_count = 0
Kp = [0, 0, 0]
Kd = [0, 0, 0]

# udp = sdk.UDP(8080, "192.168.123.10", 8007, 614, 807, False, sdk.RecvEnum.nonBlock)
udp = sdk.UDP(LOWLEVEL, 8080, "192.168.123.10", 8007)
safe = sdk.Safety(sdk.LeggedType.B1)

cmd = sdk.LowCmd()
state = sdk.LowState()
udp.InitCmdData(cmd)

UDP Initialized. socketfd: 67   Port: 8080


In [6]:

import socket
import struct
import threading
import time
class AtiFTSensor:
    def __init__(self):
        self.initialized_ = False
        self.going_ = False
        self.streaming_ = False
        self.F_bias_ = [0.0] * 3
        self.T_bias_ = [0.0] * 3
        self.F_ = [0.0] * 3
        self.T_ = [0.0] * 3
        self.rdt_sequence_ = 0
        self.ft_sequence_ = 0
        self.status_ = 0
        self.count_per_force_ = 1000000.0
        self.count_per_torque_ = 1000000.0
        self.force_torque_ = [0.0] * 6
        self.mutex_ = threading.Lock()
        self.socket_ = None
        self.local_address_ = ('', 49152)
        self.remote_address_ = ('192.168.4.1', 49152)

    def initialize(self):
        if self.initialized_:
            print("warning already initialized")
            return True
        print("initializing")
        
        try:
            self.socket_ = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            self.socket_.bind(self.local_address_)
            self.socket_.connect(self.remote_address_)
        except socket.error as e:
            print(f"Error: {e}")
            return False

        print("created sockets")
        self.initialized_ = True
        return self.initialized_

    def read_ft(self):
        msg = struct.pack('>HHH', 0x1234, 0x0002, 0)
        self.socket_.send(msg)

        internal_going = True

        while internal_going:
            data = self.socket_.recv(36)  # Assuming message size is 48 bytes
            if len(data) != 36:
                print(f"Received message of unexpected length {len(data)}")

            self.mutex_.acquire()
            # print(data)
            # breakpoint()
            (self.rdt_sequence_, self.ft_sequence_, self.status_, 
             self.Fx, self.Fy, self.Fz, 
             self.Tx, self.Ty, self.Tz) = struct.unpack('>3I6i', data)
            
            
            self.F_[0] = self.Fx / self.count_per_force_ - self.F_bias_[0]
            self.F_[1] = self.Fy / self.count_per_force_ - self.F_bias_[1]
            self.F_[2] = self.Fz / self.count_per_force_ - self.F_bias_[2]
            self.T_[0] = self.Tx / self.count_per_torque_ - self.T_bias_[0]
            self.T_[1] = self.Ty / self.count_per_torque_ - self.T_bias_[1]
            self.T_[2] = self.Tz / self.count_per_torque_ - self.T_bias_[2]

            if self.streaming_:
                # Implement streaming logic here
                pass

            internal_going = self.going_
            self.mutex_.release()

    def set_bias(self, force=None, torque=None):
        self.mutex_.acquire()
        for i in range(3):
            if force is None:
                self.F_bias_[i] = self.F_[i]
            else:
                self.F_bias_[i] = force[i]

            if torque is None:
                self.T_bias_[i] = self.T_[i]
            else:
                self.T_bias_[i] = torque[i]
        self.mutex_.release()

    def reset_bias(self):
        self.mutex_.acquire()
        self.F_bias_ = [0.0] * 3
        self.T_bias_ = [0.0] * 3
        self.mutex_.release()

    def get_status(self):
        self.mutex_.acquire()
        status = (self.rdt_sequence_, self.ft_sequence_, self.status_)
        self.mutex_.release()
        return status

    def get_ft(self):
        self.mutex_.acquire()
        ft = (self.F_[:], self.T_[:])
        self.mutex_.release()
        return ft

    def get_ft_vector(self):
        self.mutex_.acquire()
        ft_vector = self.F_ + self.T_
        self.mutex_.release()
        return ft_vector

    def stop(self):
        if self.initialized_:
            self.mutex_.acquire()
            self.going_ = False
            self.mutex_.release()
            self.socket_.close()
            self.initialized_ = False

    def stream(self, stream):
        self.mutex_.acquire()
        self.streaming_ = stream
        self.mutex_.release()

    def __del__(self):
        self.stop()


In [ ]:
sensor = AtiFTSensor()
sensor.initialize()


In [ ]:
sensor.read_ft() 
f, t = sensor.get_ft() # R
f[2]

In [ ]:
fz_bias = 0.0
for i in range(1000):
    time.sleep(0.001)
    sensor.read_ft() 
    f, t = sensor.get_ft()
    fz_bias +=     f[2]

fz_bias /=1000
    

In [28]:
import pinocchio as pin
import numpy as np
def updateModel(state, model):
    q = np.array([m.q for m in state.motorState])
    dq = np.array([m.dq for m in state.motorState])
    quat = np.array(state.imu.quaternion)
    print(quat)
    R = pin.Quaternion(quat).matrix()
    T = np.eye(4)
    T[0:3,0:3]=R
    model.updateAllPose(T = np.eye(4), q = np.zeros(12), dq = np.zeros(12), v= np.zeros(6))
    return model.getInfo()

    
updateModel(state, model)
model.updateAllPose

[0. 0. 0. 0.]


NameError: name 'q_' is not defined

In [ ]:
Tpi = 0
motiontime = 0

data_force = []
data_error = []

while True:
    time.sleep(0.002)
    motiontime += 1
    udp.Recv()
    udp.GetRecv(state)
    # print(state.motorState[d['FR_2']].q)
    if( motiontime >= 0):

        # first, get record initial position
        if( motiontime >= 0 and motiontime < 10):
            qInit[0] = state.motorState[d['FR_0']].q
            qInit[1] = state.motorState[d['FR_1']].q
            qInit[2] = state.motorState[d['FR_2']].q
    
    cmd.motorCmd[d['FR_0']].q = 0.
    cmd.motorCmd[d['FR_0']].dq = 0
    cmd.motorCmd[d['FR_0']].Kp = 0.
    cmd.motorCmd[d['FR_0']].Kd = 0.
    cmd.motorCmd[d['FR_0']].tau = 0.

    cmd.motorCmd[d['FR_1']].q = 0.
    cmd.motorCmd[d['FR_1']].dq = 0.
    cmd.motorCmd[d['FR_1']].Kp = 0.
    cmd.motorCmd[d['FR_1']].Kd = 0.
    cmd.motorCmd[d['FR_1']].tau = 0.0

    cmd.motorCmd[d['FR_2']].q =  -1.5250451564788818
    cmd.motorCmd[d['FR_2']].dq = 0
    cmd.motorCmd[d['FR_2']].Kp = 50.
    cmd.motorCmd[d['FR_2']].Kd = 0.
    cmd.motorCmd[d['FR_2']].tau = 0.0

    error = (-1.5250451564788818 - state.motorState[d['FR_2']].q)
    data_error.append(error)

    sensor.read_ft() 
    f, t = sensor.get_ft()
    data_force.append(f[2] - fz_bias)
    udp.SetSend(cmd)
    udp.Send()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data_error = np.array(data_error)
data_force = np.array(data_force)



In [ ]:
import pickle
with open('b1_knee_joint_compliance_push.pkl', 'wb') as f:
    pickle.dump({
        'arm_length':0.23,
        'P':50,
        'angular_error':data_error,
        'ATI_Fz':data_force
    }, f)

In [ ]:
%matplotlib
plt.plot(data_error, 0.23*data_force)
plt.grid(True)
plt.ylabel('Torque(N.m)')
plt.xlabel('Error(Rad)')
plt.savefig('B1_knee_compliance.pdf')